In [36]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import SystemMessage
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

True

In [37]:
llm = ChatOllama(model="llama3.1:latest", temperature=0)

In [38]:
import aiohttp
import os

async def search(query: str, **kwargs):
    params = {
        "q": query,
        **{key: value for key, value in kwargs.items() if value is not None}
    }
    headers = {
        'X-API-KEY': os.environ['SERPER_API_KEY'],
        'Content-Type': 'application/json'
    }
    async with aiohttp.ClientSession() as session:
        # Pass ssl=False directly to the request
        async with session.post("https://google.serper.dev/search", ssl=False, params=params, headers=headers) as response:
            return await response.json()

In [39]:
@tool
async def search_google(query: str):
    """ Searches for relevant information online. """
    return await search(query)

In [40]:
agent = create_agent(
    model=llm,
    tools=[search_google],
    system_prompt=SystemMessage(content="You are an online search assitant."),
    debug=True
)

In [41]:
user_query = {'messages': [{'role': 'user', 'content': 'Who is the Cheif Minister of West Bengal?'}]}
resp = await agent.ainvoke(input=user_query)

[values] {'messages': [HumanMessage(content='Who is the Cheif Minister of West Bengal?', additional_kwargs={}, response_metadata={}, id='0104068b-adad-48e6-ad6e-7ff739207334')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-05-11T08:14:13.219351Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2697517750, 'load_duration': 800866917, 'prompt_eval_count': 162, 'prompt_eval_duration': 944003042, 'eval_count': 21, 'eval_duration': 948621835, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--019e1619-ce57-79b1-860c-eea4f23cca12-0', tool_calls=[{'name': 'search_google', 'args': {'query': 'Chief Minister of West Bengal'}, 'id': '7a946a70-735d-456e-a108-37befcb50cae', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 162, 'output_tokens': 21, 'total_tokens': 183})]}}
[values] {'messages': [HumanMessage(content='Who is 

In [44]:
resp["messages"][-1].content

'The current Chief Minister of West Bengal is Suvendu Adhikari. He was sworn in as the 9th Chief Minister of West Bengal on May 9, 2026, and is the first Chief Minister from the Bharatiya Janata Party (BJP) in the state.'